In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

In [14]:
spark = SparkSession.builder.appName("chapter2").getOrCreate()

In [30]:
flightSchema = StructType(
    [
        StructField("DEST_COUNTRY_NAME",dataType=StringType(),nullable=True),
        StructField("ORIGIN_COUNTRY_NAME",dataType=StringType(),nullable=True),
        StructField("COUNT",dataType=IntegerType(),nullable=True),
    ]
)

In [46]:
flightData2015 = spark\
    .read\
    .csv("./Spark-The-Definitive-Guide/data/flight-data/csv/2015-summary.csv", header=True, schema= flightSchema)

In [47]:
flightData2015.explain()

== Physical Plan ==
FileScan csv [DEST_COUNTRY_NAME#1042,ORIGIN_COUNTRY_NAME#1043,COUNT#1044] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/talha/Learning/apacheSpark/Spark-The-Definitive-Guide/data..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,COUNT:int>




In [51]:
flightData2015.sort("COUNT").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [COUNT#1044 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(COUNT#1044 ASC NULLS FIRST, 5), ENSURE_REQUIREMENTS, [plan_id=89]
      +- FileScan csv [DEST_COUNTRY_NAME#1042,ORIGIN_COUNTRY_NAME#1043,COUNT#1044] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/talha/Learning/apacheSpark/Spark-The-Definitive-Guide/data..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,COUNT:int>




In [52]:
flightData2015.sort("COUNT").take(3)

[Row(DEST_COUNTRY_NAME='Moldova', ORIGIN_COUNTRY_NAME='United States', COUNT=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', COUNT=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Singapore', COUNT=1)]

In [50]:
spark.conf.set("spark.sql.shuffle.partitions",5)

In [54]:
flightData2015.sort("COUNT").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [COUNT#1044 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(COUNT#1044 ASC NULLS FIRST, 5), ENSURE_REQUIREMENTS, [plan_id=111]
      +- FileScan csv [DEST_COUNTRY_NAME#1042,ORIGIN_COUNTRY_NAME#1043,COUNT#1044] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/talha/Learning/apacheSpark/Spark-The-Definitive-Guide/data..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,COUNT:int>




In [57]:
flightData2015.createOrReplaceTempView("flightData2015")

In [60]:
sqlWay = spark.sql(
    """
    SELECT DISTINCT DEST_COUNTRY_NAME, COUNT(1)
    FROM flightData2015
    GROUP BY 1
    """
)

In [61]:
sqlWay.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[DEST_COUNTRY_NAME#1042], functions=[count(1)])
   +- Exchange hashpartitioning(DEST_COUNTRY_NAME#1042, 5), ENSURE_REQUIREMENTS, [plan_id=140]
      +- HashAggregate(keys=[DEST_COUNTRY_NAME#1042], functions=[partial_count(1)])
         +- FileScan csv [DEST_COUNTRY_NAME#1042] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/talha/Learning/apacheSpark/Spark-The-Definitive-Guide/data..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string>




In [62]:
dataFrameWay = flightData2015.groupby("DEST_COUNTRY_NAME").count()

In [63]:
dataFrameWay.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[DEST_COUNTRY_NAME#1042], functions=[count(1)])
   +- Exchange hashpartitioning(DEST_COUNTRY_NAME#1042, 5), ENSURE_REQUIREMENTS, [plan_id=153]
      +- HashAggregate(keys=[DEST_COUNTRY_NAME#1042], functions=[partial_count(1)])
         +- FileScan csv [DEST_COUNTRY_NAME#1042] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/talha/Learning/apacheSpark/Spark-The-Definitive-Guide/data..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string>




In [76]:
spark.sql(
    """
    SELECT
        MAX(COUNT)
    FROM 
        flightData2015
    """
).show(1)

+----------+
|max(COUNT)|
+----------+
|    370002|
+----------+



In [71]:
from pyspark.sql.functions import max

flightData2015.select(max("COUNT").alias("Max Flights")).show(1)

+-----------+
|Max Flights|
+-----------+
|     370002|
+-----------+



In [82]:
# Top 5 destination countries
maxSql = spark.sql(
    """
        SELECT DISTINCT DEST_COUNTRY_NAME, SUM(COUNT) as destinationTotal
        FROM flightData2015
        GROUP BY 1
        ORDER BY 2 desc
        LIMIT 5
    
    """
)

maxSql.show()


+-----------------+----------------+
|DEST_COUNTRY_NAME|destinationTotal|
+-----------------+----------------+
|    United States|          411352|
|           Canada|            8399|
|           Mexico|            7140|
|   United Kingdom|            2025|
|            Japan|            1548|
+-----------------+----------------+



In [95]:
from pyspark.sql.functions import sum, desc
maxDF = flightData2015\
    .groupBy("DEST_COUNTRY_NAME")\
    .agg(sum("COUNT").alias("destinationTotal"))\
    .orderBy(desc(sum("COUNT").alias("destinationTotal")))\
    .limit(5)

maxDF.show()

+-----------------+----------------+
|DEST_COUNTRY_NAME|destinationTotal|
+-----------------+----------------+
|    United States|          411352|
|           Canada|            8399|
|           Mexico|            7140|
|   United Kingdom|            2025|
|            Japan|            1548|
+-----------------+----------------+



In [112]:
maxDF2 = flightData2015.groupBy("DEST_COUNTRY_NAME","ORIGIN_COUNTRY_NAME")\
    .sum("COUNT")\
    .withColumnRenamed("sum(COUNT)","destinationTotal")\
    .orderBy(desc("destinationTotal"))\
    .where("DEST_COUNTRY_NAME like '%United%'")\
    .where("ORIGIN_COUNTRY_NAME = 'India'")\
    .limit(5)
maxDF2.show()

+-----------------+-------------------+----------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|destinationTotal|
+-----------------+-------------------+----------------+
|    United States|              India|              62|
+-----------------+-------------------+----------------+



In [115]:
# maxSql.explain()
# maxDF.explain(extended=True)
maxDF2.explain(mode="formatted")
# maxDF2.explain()

== Physical Plan ==
AdaptiveSparkPlan (7)
+- TakeOrderedAndProject (6)
   +- HashAggregate (5)
      +- Exchange (4)
         +- HashAggregate (3)
            +- Filter (2)
               +- Scan csv  (1)


(1) Scan csv 
Output [3]: [DEST_COUNTRY_NAME#1042, ORIGIN_COUNTRY_NAME#1043, COUNT#1044]
Batched: false
Location: InMemoryFileIndex [file:/Users/talha/Learning/apacheSpark/Spark-The-Definitive-Guide/data/flight-data/csv/2015-summary.csv]
PushedFilters: [IsNotNull(DEST_COUNTRY_NAME), IsNotNull(ORIGIN_COUNTRY_NAME), StringContains(DEST_COUNTRY_NAME,United), EqualTo(ORIGIN_COUNTRY_NAME,India)]
ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,COUNT:int>

(2) Filter
Input [3]: [DEST_COUNTRY_NAME#1042, ORIGIN_COUNTRY_NAME#1043, COUNT#1044]
Condition : (((isnotnull(DEST_COUNTRY_NAME#1042) AND isnotnull(ORIGIN_COUNTRY_NAME#1043)) AND Contains(DEST_COUNTRY_NAME#1042, United)) AND (ORIGIN_COUNTRY_NAME#1043 = India))

(3) HashAggregate
Input [3]: [DEST_COUNTRY_NAME#1042, 